# TEMPO vs TROPOMI Tropospheric NO2 Comparison

In [ ]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress, pearsonr
from IPython.display import Image, display

# First run only: ee.Authenticate() opens a browser consent screen and caches credentials locally.
# ee.Authenticate()
ee.Initialize(project='applied-remote-sensing-485001')

In [ ]:
study_area = ee.Geometry.Rectangle([-91.5, 29.5, -90.0, 31.0])
palette = ['440154', '482878', '3e4989', '31688e', '26828e',
           '1f9e89', '35b779', '6ece58', 'b5de2b', 'fde725']

## Helper functions

In [ ]:
def mask_tropomi(img):
    mask = img.select('cloud_fraction').lt(0.5)
    return img.updateMask(mask).select('tropospheric_NO2_column_number_density')


def aggregate_tempo(tempo_img, target_proj):
    return (tempo_img
            .setDefaultProjection(crs='EPSG:4326', scale=2000)
            .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=256)
            .reproject(crs=target_proj, scale=5500))


def normalize_img(img, band_name, geometry):
    stats = img.reduceRegion(reducer=ee.Reducer.minMax(), geometry=geometry,
                              scale=2000, maxPixels=1e9)
    lo = ee.Number(stats.get(band_name + '_min'))
    hi = ee.Number(stats.get(band_name + '_max'))
    return img.subtract(lo).divide(hi.subtract(lo))


def fc_to_df(fc, columns=None):
    features = fc.getInfo()['features']
    df = pd.DataFrame([f['properties'] for f in features])
    return df[columns] if columns else df

## Shared TROPOMI projection

In [ ]:
tropomi_proj = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2')
                .filterDate('2024-11-01', '2024-11-30')
                .first()
                .select('tropospheric_NO2_column_number_density')
                .projection())

## Composite caching (Earth Engine Assets)

In [ ]:
import time

ASSET_FOLDER = 'projects/applied-remote-sensing-485001/assets/tempo_tropomi_cache'

try:
    ee.data.getAsset(ASSET_FOLDER)
except ee.EEException:
    ee.data.createAsset({'type': 'Folder'}, ASSET_FOLDER)
    print('Created asset folder:', ASSET_FOLDER)


def get_or_export(img, name, scale, crs='EPSG:4326'):
    """Return the cached asset for `img`, exporting it first if it doesn't exist yet."""
    asset_id = f'{ASSET_FOLDER}/{name}'
    try:
        ee.data.getAsset(asset_id)
        return ee.Image(asset_id)
    except ee.EEException:
        pass

    task = ee.batch.Export.image.toAsset(
        image=img.clip(study_area),
        description=name,
        assetId=asset_id,
        region=study_area,
        scale=scale,
        crs=crs,
        maxPixels=1e9,
    )
    task.start()
    print(f'Exporting {name} (one-time)...', end='', flush=True)
    while True:
        state = task.status()['state']
        if state == 'COMPLETED':
            print(' done.')
            return ee.Image(asset_id)
        if state in ('FAILED', 'CANCELLED'):
            raise RuntimeError(f'Export {name} {state}: {task.status()}')
        print('.', end='', flush=True)
        time.sleep(10)

## Scatter plot helper

In [ ]:
def plot_scatter(df, title, color, xmax=70, ymax=10):
    x, y = df['tempo'].values, df['tropomi'].values
    slope, intercept, r, p, se = linregress(x, y)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(x, y, s=20, alpha=0.6, color=color)
    xs = np.linspace(0, xmax, 100)
    ax.plot(xs, slope * xs + intercept, color='black', linewidth=1, alpha=0.8,
            label=f'R\u00b2 = {r**2:.3f}')
    ax.set_xlim(0, xmax)
    ax.set_ylim(0, ymax)
    ax.set_xlabel('TEMPO (\u00d710\u00b9\u2074 molecules/cm\u00b2)')
    ax.set_ylabel('TROPOMI (\u00d710\u00b9\u2074 molecules/cm\u00b2)')
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()
    return slope, intercept, r, p

## Single day (Nov 15, 2024)

In [ ]:
tempo_day = (ee.ImageCollection('NASA/TEMPO/NO2_L3_QA')
             .filterDate('2024-11-15T18:00', '2024-11-15T19:00')
             .filterBounds(study_area)
             .select('vertical_column_troposphere')
             .mean())

tropomi_day = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2')
               .filterDate('2024-11-15', '2024-11-16')
               .filterBounds(study_area)
               .map(mask_tropomi)
               .mean()
               .multiply(6.022e19)
               .reproject(crs=tropomi_proj, scale=5500))

tempo_day_agg = aggregate_tempo(tempo_day, tropomi_proj)

sample_day_fc = (tempo_day_agg.divide(1e14).rename('tempo')
                  .addBands(tropomi_day.divide(1e14).rename('tropomi'))
                  .sample(region=study_area, scale=5500, numPixels=500))

sample_day = fc_to_df(sample_day_fc, columns=['tempo', 'tropomi'])
plot_scatter(sample_day, 'Matched TEMPO (~18:30 UTC) vs TROPOMI \u2014 Nov 15 2024', color='#e377c2')

## Monthly aggregate (Nov 2024)

In [ ]:
tempo_monthly = (ee.ImageCollection('NASA/TEMPO/NO2_L3_QA')
                  .filterDate('2024-11-01', '2024-11-30')
                  .filterBounds(study_area)
                  .mean()
                  .select('vertical_column_troposphere'))

tropomi_monthly = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2')
                    .filterDate('2024-11-01', '2024-11-30')
                    .filterBounds(study_area)
                    .map(mask_tropomi)
                    .mean()
                    .multiply(6.022e19)
                    .reproject(crs=tropomi_proj, scale=5500))

tempo_monthly_agg = aggregate_tempo(tempo_monthly, tropomi_proj)

sample_monthly_fc = (tempo_monthly_agg.divide(1e14).rename('tempo')
                      .addBands(tropomi_monthly.divide(1e14).rename('tropomi'))
                      .sample(region=study_area, scale=5500, numPixels=500))

sample_monthly = fc_to_df(sample_monthly_fc, columns=['tempo', 'tropomi'])
plot_scatter(sample_monthly, 'Monthly Mean TEMPO vs TROPOMI Tropospheric NO\u2082 (Nov 2024)', color='#1f77b4')

## Annual aggregate (2024)

In [ ]:
tempo_annual = (ee.ImageCollection('NASA/TEMPO/NO2_L3_QA')
                .filterDate('2024-01-01', '2024-12-31')
                .filterBounds(study_area)
                .mean()
                .select('vertical_column_troposphere'))

tropomi_annual = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2')
                   .filterDate('2024-01-01', '2024-12-31')
                   .filterBounds(study_area)
                   .map(mask_tropomi)
                   .mean()
                   .multiply(6.022e19)
                   .reproject(crs=tropomi_proj, scale=5500))

tempo_annual = get_or_export(tempo_annual, 'tempo_annual_2024', scale=2000)
tropomi_annual = get_or_export(tropomi_annual, 'tropomi_annual_2024', scale=5500)

tempo_annual_agg = aggregate_tempo(tempo_annual, tropomi_proj)

sample_annual_fc = (tempo_annual_agg.divide(1e14).rename('tempo')
                     .addBands(tropomi_annual.divide(1e14).rename('tropomi'))
                     .sample(region=study_area, scale=5500, numPixels=500))

sample_annual = fc_to_df(sample_annual_fc, columns=['tempo', 'tropomi'])
plot_scatter(sample_annual, 'Annual Mean TEMPO vs TROPOMI Tropospheric NO₂ (2024)', color='#2ca02c')

## Normalized differences

In [ ]:
tropomi_day_at_tempo_res = tropomi_day.reproject(crs='EPSG:4326', scale=2000)
tempo_day_scaled = normalize_img(tempo_day, 'vertical_column_troposphere', study_area)
tropomi_day_scaled = normalize_img(tropomi_day_at_tempo_res, 'tropospheric_NO2_column_number_density', study_area)
diff_norm_day = tempo_day_scaled.subtract(tropomi_day_scaled)

tropomi_monthly_at_tempo_res = tropomi_monthly.reproject(crs='EPSG:4326', scale=2000)
tempo_monthly_scaled = normalize_img(tempo_monthly, 'vertical_column_troposphere', study_area)
tropomi_monthly_scaled = normalize_img(tropomi_monthly_at_tempo_res, 'tropospheric_NO2_column_number_density', study_area)
diff_norm_monthly = tempo_monthly_scaled.subtract(tropomi_monthly_scaled)

tropomi_annual_at_tempo_res = tropomi_annual.reproject(crs='EPSG:4326', scale=2000)
tempo_annual_scaled = normalize_img(tempo_annual, 'vertical_column_troposphere', study_area)
tropomi_annual_scaled = normalize_img(tropomi_annual_at_tempo_res, 'tropospheric_NO2_column_number_density', study_area)
diff_norm_annual = tempo_annual_scaled.subtract(tropomi_annual_scaled)

## Interactive map

In [ ]:
Map = geemap.Map()
Map.centerObject(study_area, 8)

norm_vis = {'min': 0, 'max': 1, 'palette': palette}
diff_vis = {'min': -1, 'max': 1, 'palette': ['blue', 'white', 'red']}

Map.addLayer(ee.Image(1).clip(study_area), {'opacity': 0.35, 'palette': ['red']}, 'Study Area')

Map.addLayer(tropomi_day_scaled.clip(study_area), norm_vis, 'Nov 15 | TROPOMI NO\u2082 (normalized)')
Map.addLayer(tempo_day_scaled.clip(study_area), norm_vis, 'Nov 15 | TEMPO NO\u2082 (normalized)')

Map.addLayer(tropomi_monthly_scaled.clip(study_area), norm_vis, 'Nov 2024 | TROPOMI NO\u2082 (normalized)')
Map.addLayer(tempo_monthly_scaled.clip(study_area), norm_vis, 'Nov 2024 | TEMPO NO\u2082 (normalized)')

Map.addLayer(tropomi_annual_scaled.clip(study_area), norm_vis, '2024 Annual | TROPOMI NO\u2082 (normalized)')
Map.addLayer(tempo_annual_scaled.clip(study_area), norm_vis, '2024 Annual | TEMPO NO\u2082 (normalized)')

Map.addLayer(diff_norm_day.clip(study_area), diff_vis, 'Nov 15 | Normalized difference (TEMPO \u2212 TROPOMI)')
Map.addLayer(diff_norm_monthly.clip(study_area), diff_vis, 'Nov 2024 | Normalized difference (TEMPO \u2212 TROPOMI)')
Map.addLayer(diff_norm_annual.clip(study_area), diff_vis, '2024 Annual | Normalized difference (TEMPO \u2212 TROPOMI)')

Map

## Hourly time series (June 15, 2024)

In [ ]:
import matplotlib.dates as mdates

hourly_ic = (ee.ImageCollection('NASA/TEMPO/NO2_L3_QA')
             .filterDate('2024-06-15T05:00', '2024-06-16T05:00')
             .filterBounds(study_area)
             .select('vertical_column_troposphere'))


def with_region_mean(img):
    stat = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=study_area,
                             scale=2000, maxPixels=1e9)
    return img.set('mean_no2', stat.get('vertical_column_troposphere'))


hourly_with_stats = hourly_ic.map(with_region_mean)

# Pull time + mean as paired rows in one call — a fully-masked scene drops its
# 'mean_no2' property, which desyncs two separate aggregate_array() calls.
rows = hourly_with_stats.reduceColumns(
    reducer=ee.Reducer.toList(2),
    selectors=['system:time_start', 'mean_no2']
).get('list').getInfo()

LOCAL_TZ = 'America/Chicago'  # Louisiana Gulf Coast study area

hourly_df = pd.DataFrame(rows, columns=['time_ms', 'no2'])
hourly_df['time'] = pd.to_datetime(hourly_df['time_ms'], unit='ms', utc=True).dt.tz_convert(LOCAL_TZ)
hourly_df['no2'] = pd.to_numeric(hourly_df['no2'], errors='coerce') / 1e14
hourly_df = hourly_df.dropna(subset=['no2']).sort_values('time')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(hourly_df['time'], hourly_df['no2'], color='#1f77b4', linewidth=1, linestyle='--')
ax.scatter(hourly_df['time'], hourly_df['no2'], color='#1f77b4', s=40)
ax.set_xlabel('Time (Local, Central)')
ax.set_ylabel('NO₂ (×10¹⁴ molecules/cm²)')
ax.set_title('TEMPO Hourly NO₂ — June 15 2024 (Local Time)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%I:%M %p', tz=hourly_df['time'].dt.tz))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
scan_times = (ee.ImageCollection('NASA/TEMPO/NO2_L3_QA')
              .filterDate('2024-06-15', '2024-06-16')
              .filterBounds(study_area)
              .aggregate_array('system:time_start')
              .getInfo())
print('June 15 scan times (local):',
      [pd.to_datetime(t, unit='ms', utc=True).tz_convert(LOCAL_TZ).strftime('%I:%M %p') for t in scan_times])

## Summary statistics

In [ ]:
def print_diff_stats(label, diff_img, band='vertical_column_troposphere', scale=2000):
    reducer = (ee.Reducer.mean()
               .combine(ee.Reducer.stdDev(), sharedInputs=True)
               .combine(ee.Reducer.percentile([5, 50, 95]), sharedInputs=True))
    stats = diff_img.reduceRegion(reducer=reducer, geometry=study_area,
                                   scale=scale, maxPixels=1e9).getInfo()
    print(f"{label} | mean: {stats[band + '_mean']:.3f} "
          f"| sd: {stats[band + '_stdDev']:.3f} "
          f"| p5: {stats[band + '_p5']:.3f} "
          f"| p50: {stats[band + '_p50']:.3f} "
          f"| p95: {stats[band + '_p95']:.3f}")

    frac = diff_img.gt(0).reduceRegion(reducer=ee.Reducer.mean(), geometry=study_area,
                                        scale=scale, maxPixels=1e9).getInfo()
    print(f"{label} | TEMPO > TROPOMI: {frac[band] * 100:.1f}%")


def print_corr(label, df):
    slope, intercept, r, p, se = linregress(df['tempo'], df['tropomi'])
    print(f"{label} | scale: {slope:.4f}, offset: {intercept:.4f}")
    print(f"{label} | R: {r:.4f} (p = {p:.2e})")

In [ ]:
print_diff_stats('Nov 15 2024   ', diff_norm_day)
print_corr('Nov 15 2024   ', sample_day)

print_diff_stats('Nov 2024      ', diff_norm_monthly)
print_corr('Nov 2024      ', sample_monthly)

print_diff_stats('2024 Annual   ', diff_norm_annual)
print_corr('2024 Annual   ', sample_annual)

## Thumbnails

In [ ]:
import io
import time
import urllib.request
from PIL import Image as PILImage


def fetch_thumb(img, vis_params, dimensions=400, retries=4):
    url = img.getThumbURL({**vis_params, 'region': study_area, 'dimensions': dimensions, 'format': 'png'})
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=60) as resp:
                return PILImage.open(io.BytesIO(resp.read()))
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)


thumb_norm = {'min': 0, 'max': 1, 'palette': palette}
thumb_diff = {'min': -1, 'max': 1, 'palette': ['blue', 'white', 'red']}
col_titles = ['TROPOMI (normalized)', 'TEMPO (normalized)', 'Diff (TEMPO − TROPOMI)']


def show_thumb_row(row_label, tropomi_img, tempo_img, diff_img):
    thumbs = [
        fetch_thumb(tropomi_img.clip(study_area), thumb_norm),
        fetch_thumb(tempo_img.clip(study_area), thumb_norm),
        fetch_thumb(diff_img.clip(study_area), thumb_diff),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
    for ax, thumb, title in zip(axes, thumbs, col_titles):
        ax.imshow(thumb)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(title)
    axes[0].set_ylabel(row_label, fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
show_thumb_row('Nov 15', tropomi_day_scaled, tempo_day_scaled, diff_norm_day)

In [ ]:
show_thumb_row('Nov 2024', tropomi_monthly_scaled, tempo_monthly_scaled, diff_norm_monthly)

In [ ]:
show_thumb_row('2024 Annual', tropomi_annual_scaled, tempo_annual_scaled, diff_norm_annual)

## Raw value diagnostics

In [ ]:
# Filter to images with unmasked pixel data in the region to see the true overpass(es).

def filter_to_valid_data(collection, geometry, band, scale=5500):
    def add_valid_count(img):
        count = img.select(band).reduceRegion(
            reducer=ee.Reducer.count(), geometry=geometry, scale=scale, maxPixels=1e9
        ).get(band)
        return img.set('valid_count', count)
    return collection.map(add_valid_count).filter(ee.Filter.gt('valid_count', 0))


tropomi_nov15_all = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2')
                      .filterDate('2024-11-15', '2024-11-16')
                      .filterBounds(study_area)
                      .map(mask_tropomi))

tropomi_nov15_valid = filter_to_valid_data(
    tropomi_nov15_all, study_area, 'tropospheric_NO2_column_number_density')

print('TROPOMI Nov 15 footprint-intersecting images (misleading — includes empty grazes):',
      tropomi_nov15_all.size().getInfo())

print('TROPOMI Nov 15 actual overpasses (valid data in study area):',
      [pd.to_datetime(t, unit='ms', utc=True).strftime('%Y-%m-%d %H:%M') for t in
       tropomi_nov15_valid.aggregate_array('system:time_start').getInfo()])

print('TROPOMI Nov 15 image count with valid data:', tropomi_nov15_valid.size().getInfo())

In [ ]:
raw_tropomi = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2')
               .filterDate('2024-11-01', '2024-11-30')
               .filterBounds(study_area)
               .map(mask_tropomi)
               .mean())

print('Raw TROPOMI mean (mol/m\u00b2):',
      raw_tropomi.reduceRegion(reducer=ee.Reducer.mean(), geometry=study_area,
                                scale=5500, maxPixels=1e9).getInfo())

print('Raw TEMPO mean (molecules/cm\u00b2):',
      tempo_monthly_agg.reduceRegion(reducer=ee.Reducer.mean(), geometry=study_area,
                                      scale=5500, maxPixels=1e9).getInfo())

print('TROPOMI \u00d710\u00b9\u2074 stats:',
      tropomi_monthly.divide(1e14).reduceRegion(
          reducer=ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True),
          geometry=study_area, scale=5500, maxPixels=1e9).getInfo())

print('TROPOMI converted mean:',
      tropomi_monthly.reduceRegion(reducer=ee.Reducer.mean(), geometry=study_area,
                                    scale=5500, maxPixels=1e9).getInfo())

## EPA AQS ground monitors (hourly NO₂, 2024)

Ground-truth data from the EPA Air Quality System API — hourly NO₂ (parameter `42602`) for all monitors in the study area. 8 active monitors span urban core (Capitol), near-road (I-610), the industrial corridor (Dutchtown, Bayou Plaquemine, Port Allen), and rural background (French Settlement, Pride).

Fetched once and cached to `data/aqs_no2_2024.csv`. Validation uses correlation and diurnal-pattern agreement, not absolute values.

In [ ]:
import json
import requests
from pathlib import Path

AQS_BASE = 'https://aqs.epa.gov/data/api'
BBOX = {'minlat': 29.5, 'maxlat': 31.0, 'minlon': -91.5, 'maxlon': -90.0}
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

with open('aqs_config.json') as f:
    _aqs = json.load(f)


def aqs_get(endpoint, **params):
    r = requests.get(f'{AQS_BASE}/{endpoint}', params={
        'email': _aqs['email'], 'key': _aqs['key'], **BBOX, **params,
    }, timeout=120)
    r.raise_for_status()
    payload = r.json()
    status = payload['Header'][0]['status']
    if status != 'Success':
        raise RuntimeError(f'AQS {endpoint} returned: {status}')
    return pd.DataFrame(payload['Data'])


def fetch_aqs_no2(year=2024):
    """Hourly NO2 for all monitors in the study area, cached to CSV after first fetch."""
    cache = DATA_DIR / f'aqs_no2_{year}.csv'
    if cache.exists():
        return pd.read_csv(cache, parse_dates=['datetime_gmt'])

    # Site names come from the monitors endpoint — sampleData rows don't carry them.
    monitors = aqs_get('monitors/byBox', param='42602',
                       bdate=f'{year}0101', edate=f'{year}1231')
    monitors['site_id'] = (monitors['state_code'] + '-' + monitors['county_code']
                           + '-' + monitors['site_number'])
    site_names = monitors.drop_duplicates('site_id').set_index('site_id')['local_site_name']

    frames = []
    for m in range(1, 13):
        bdate = f'{year}{m:02d}01'
        edate = (pd.Timestamp(year=year, month=m, day=1) + pd.offsets.MonthEnd(0)).strftime('%Y%m%d')
        frames.append(aqs_get('sampleData/byBox', param='42602', bdate=bdate, edate=edate))
        print(f'{year}-{m:02d}: {len(frames[-1])} rows')
        time.sleep(6)  # AQS asks for <=10 requests/minute

    df = pd.concat(frames, ignore_index=True)
    df['site_id'] = df['state_code'] + '-' + df['county_code'] + '-' + df['site_number']
    df['local_site_name'] = df['site_id'].map(site_names)
    df['datetime_gmt'] = pd.to_datetime(df['date_gmt'] + ' ' + df['time_gmt'], utc=True)
    df = (df[['site_id', 'local_site_name', 'latitude', 'longitude',
              'datetime_gmt', 'sample_measurement', 'units_of_measure']]
          .dropna(subset=['sample_measurement'])
          .sort_values(['site_id', 'datetime_gmt']))
    df.to_csv(cache, index=False)
    return df


aqs = fetch_aqs_no2(2024)
print(f'\n{len(aqs):,} hourly records')

In [ ]:
# Per-site summary: completeness and NO2 levels
site_summary = (aqs.groupby(['site_id', 'local_site_name'])
                .agg(lat=('latitude', 'first'),
                     lon=('longitude', 'first'),
                     n_hours=('sample_measurement', 'size'),
                     mean_ppb=('sample_measurement', 'mean'),
                     p95_ppb=('sample_measurement', lambda s: s.quantile(0.95)))
                .round(2))
site_summary['completeness_%'] = (site_summary['n_hours'] / 8784 * 100).round(1)  # 2024 is a leap year
site_summary

# Monitors on the map for context
monitor_fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([row.lon, row.lat]), {'name': str(idx[1])})
    for idx, row in site_summary.iterrows()
])
Map.addLayer(monitor_fc, {'color': 'red'}, 'AQS NO2 monitors')
site_summary

## Satellite–monitor matchups

Three comparisons, increasing in temporal resolution:
1. **Annual site means** — satellite column (cached annual assets) vs monitor mean ppb at each of the 8 sites. Tests whether the satellites reproduce the spatial gradient (near-road > urban > industrial > rural).
2. **Hourly TEMPO at monitor points** — every 2024 TEMPO scan sampled at each monitor location (extracted month-by-month, cached to CSV).
3. **Diurnal cycle validation** — TEMPO column vs monitor ppb by local hour of day: the capability TROPOMI structurally lacks.

In [ ]:
# 1. Annual site means: satellite column vs monitor surface ppb
sites_df = site_summary.reset_index()
monitor_pts = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([r.lon, r.lat]), {'site_id': r.site_id, 'name': r.local_site_name})
    for r in sites_df.itertuples()
])

annual_at_sites = (tempo_annual.rename('tempo')
                   .addBands(tropomi_annual.rename('tropomi'))
                   .reduceRegions(collection=monitor_pts,
                                  reducer=ee.Reducer.mean(),
                                  scale=2000))

matchup_annual = fc_to_df(annual_at_sites, columns=['site_id', 'name', 'tempo', 'tropomi'])
matchup_annual[['tempo', 'tropomi']] = matchup_annual[['tempo', 'tropomi']] / 1e14
matchup_annual = matchup_annual.merge(sites_df[['site_id', 'mean_ppb']], on='site_id')

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
for ax, sat, color in [(axes[0], 'tempo', '#d62728'), (axes[1], 'tropomi', '#1f77b4')]:
    x, y = matchup_annual['mean_ppb'], matchup_annual[sat]
    slope, intercept, r, p, se = linregress(x, y)
    ax.scatter(x, y, s=60, color=color)
    for row in matchup_annual.itertuples():
        ax.annotate(row.name, (row.mean_ppb, getattr(row, sat)),
                    fontsize=8, xytext=(4, 4), textcoords='offset points')
    xs = np.linspace(x.min(), x.max(), 50)
    ax.plot(xs, slope * xs + intercept, color='black', linewidth=1,
            label=f'R = {r:.3f} (p = {p:.3f})')
    ax.set_xlabel('AQS monitor annual mean NO₂ (ppb)')
    ax.set_ylabel(f'{sat.upper()} annual mean column (×10¹⁴ molec/cm²)')
    ax.set_title(f'{sat.upper()} vs ground monitors — 2024 annual')
    ax.legend()
plt.tight_layout()
plt.show()

matchup_annual.round(2)

In [ ]:
# 2. Hourly TEMPO at monitor points — full year, extracted month-by-month with
# per-month caching so an interruption only costs the current month.
def fetch_tempo_at_monitors(year=2024):

    cache = DATA_DIR / f'tempo_at_monitors_{year}.csv'
    if cache.exists():
        return pd.read_csv(cache, parse_dates=['datetime_gmt'])

    month_frames = []
    for m in range(1, 13):
        month_cache = DATA_DIR / f'_tempo_monitors_{year}_{m:02d}.csv'
        if month_cache.exists():
            month_frames.append(pd.read_csv(month_cache))
            continue

        start = ee.Date.fromYMD(year, m, 1)
        ic = (ee.ImageCollection('NASA/TEMPO/NO2_L3_QA')
              .filterDate(start, start.advance(1, 'month'))
              .filterBounds(study_area)
              .select('vertical_column_troposphere'))

        def extract(img):
            fc = img.reduceRegions(collection=monitor_pts,
                                   reducer=ee.Reducer.mean(), scale=2000)
            return fc.map(lambda f: f.set('time', img.get('system:time_start')))

        table = (ic.map(extract).flatten()
                 .filter(ee.Filter.notNull(['mean'])))
        rows = table.reduceColumns(
            reducer=ee.Reducer.toList(3),
            selectors=['site_id', 'time', 'mean']
        ).get('list').getInfo()

        mdf = pd.DataFrame(rows, columns=['site_id', 'time_ms', 'tempo_column'])
        mdf.to_csv(month_cache, index=False)
        month_frames.append(mdf)
        print(f'{year}-{m:02d}: {len(mdf)} matchups')

    df = pd.concat(month_frames, ignore_index=True)
    df['datetime_gmt'] = pd.to_datetime(df['time_ms'], unit='ms', utc=True)
    df['tempo_column'] = df['tempo_column'] / 1e14
    df = df[['site_id', 'datetime_gmt', 'tempo_column']].sort_values(['site_id', 'datetime_gmt'])
    df.to_csv(cache, index=False)
    for m in range(1, 13):
        (DATA_DIR / f'_tempo_monitors_{year}_{m:02d}.csv').unlink(missing_ok=True)
    return df


tempo_at_monitors = fetch_tempo_at_monitors(2024)
print(f'{len(tempo_at_monitors):,} TEMPO–monitor matchups')

In [ ]:
# 3. Diurnal validation: pair each TEMPO scan with the monitor's same-hour reading,
# then compare mean diurnal cycles (normalized — column and ppb are different quantities).
tm = tempo_at_monitors.copy()
tm['hour_key'] = tm['datetime_gmt'].dt.floor('h')

aq = aqs.copy()
aq['hour_key'] = aq['datetime_gmt']  # AQS timestamps are hour-beginning

paired = tm.merge(aq[['site_id', 'hour_key', 'sample_measurement']],
                  on=['site_id', 'hour_key'], how='inner')
paired = paired.rename(columns={'sample_measurement': 'aqs_ppb'})
paired['local_hour'] = paired['datetime_gmt'].dt.tz_convert(LOCAL_TZ).dt.hour
print(f'{len(paired):,} hourly TEMPO–AQS pairs')

# Per-site correlation table
corr_rows = []
for site_id, g in paired.groupby('site_id'):
    slope, intercept, r, p, se = linregress(g['tempo_column'], g['aqs_ppb'])
    name = sites_df.loc[sites_df.site_id == site_id, 'local_site_name'].iloc[0]
    corr_rows.append({'site': name, 'n': len(g), 'R': round(r, 3),
                      'slope_ppb_per_1e14': round(slope, 3), 'p': f'{p:.1e}'})
corr_table = pd.DataFrame(corr_rows).sort_values('R', ascending=False)
display(corr_table)


In [ ]:
# TROPOMI daily overpass means over the study area, full 2024 — one valid overpass
# per day (~18:30 UTC), cached like the other extractions.
def fetch_tropomi_daily(year=2024):
    cache = DATA_DIR / f'tropomi_daily_{year}.csv'
    if cache.exists():
        return pd.read_csv(cache, parse_dates=['datetime_gmt'])

    month_frames = []
    for m in range(1, 13):
        month_cache = DATA_DIR / f'_tropomi_daily_{year}_{m:02d}.csv'
        if month_cache.exists():
            month_frames.append(pd.read_csv(month_cache))
            continue

        start = ee.Date.fromYMD(year, m, 1)
        ic = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2')
              .filterDate(start, start.advance(1, 'month'))
              .filterBounds(study_area)
              .map(mask_tropomi))

        def with_mean(img):
            stat = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=study_area,
                                     scale=5500, maxPixels=1e9)
            return img.set('no2', stat.get('tropospheric_NO2_column_number_density'))

        rows = (ic.map(with_mean)
                .filter(ee.Filter.notNull(['no2']))
                .reduceColumns(reducer=ee.Reducer.toList(2),
                               selectors=['system:time_start', 'no2'])
                .get('list').getInfo())

        mdf = pd.DataFrame(rows, columns=['time_ms', 'tropomi_molm2'])
        mdf.to_csv(month_cache, index=False)
        month_frames.append(mdf)
        print(f'{year}-{m:02d}: {len(mdf)} overpasses')

    df = pd.concat(month_frames, ignore_index=True)
    df['datetime_gmt'] = pd.to_datetime(df['time_ms'], unit='ms', utc=True)
    df['tropomi_column'] = df['tropomi_molm2'] * 6.022e19 / 1e14
    df = df[['datetime_gmt', 'tropomi_column']].sort_values('datetime_gmt')
    df.to_csv(cache, index=False)
    for m in range(1, 13):
        (DATA_DIR / f'_tropomi_daily_{year}_{m:02d}.csv').unlink(missing_ok=True)
    return df


tropomi_daily = fetch_tropomi_daily(2024)
print(f'{len(tropomi_daily):,} TROPOMI overpasses in 2024')

# Full diurnal cycle, all three instruments min-max normalized per series (0-1) so
# one axis serves all — column and surface are different quantities, and TEMPO vs
# TROPOMI carry the known ~10x scale offset, so shapes are the honest comparison.
aqs_local_hour = aqs['datetime_gmt'].dt.tz_convert(LOCAL_TZ).dt.hour
aqs_24h = aqs.groupby(aqs_local_hour)['sample_measurement'].mean()
tempo_diurnal = paired.groupby('local_hour')['tempo_column'].mean()
tropomi_diurnal = (tropomi_daily
                   .groupby(tropomi_daily['datetime_gmt'].dt.tz_convert(LOCAL_TZ).dt.hour)['tropomi_column']
                   .agg(['mean', 'size']))
print('TROPOMI overpass local hours:', dict(tropomi_diurnal['size']))


def minmax(s):
    return (s - s.min()) / (s.max() - s.min())


aqs_norm = minmax(aqs_24h)
tempo_norm = minmax(tempo_diurnal)
# TROPOMI is shown as a single timing marker: per-hour anchoring proved
# uninterpretable (few overpasses per hour bin, split across seasons by DST), so the
# diamond marks WHEN TROPOMI samples the day -- placed on the TEMPO curve, making no
# independent amplitude claim.
trop_hours_local = tropomi_daily['datetime_gmt'].dt.tz_convert(LOCAL_TZ).dt.hour \
                   + tropomi_daily['datetime_gmt'].dt.tz_convert(LOCAL_TZ).dt.minute / 60
trop_mean_hour = trop_hours_local.mean()
print(f'TROPOMI mean overpass time: {trop_mean_hour:.2f}h local')

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(aqs_norm.index, aqs_norm.values, 's-', color='#333333', label='AQS surface NO₂ (all hours)')
ax.plot(tempo_norm.index, tempo_norm.values, 'o-', color='#d62728', label='TEMPO column (daylight only)')
ax.axvline(trop_mean_hour, color='#1f77b4', linestyle=':', linewidth=2,
           label='TROPOMI overpass time (~12:30-13:30 local, one snapshot/day)')

tempo_lo, tempo_hi = tempo_diurnal.index.min(), tempo_diurnal.index.max()
trop_lo, trop_hi = tropomi_diurnal.index.min(), tropomi_diurnal.index.max()

ax.axvspan(0, tempo_lo, color='gray', alpha=0.12)
ax.axvspan(tempo_hi, 23, color='gray', alpha=0.12)

ax.annotate('No TEMPO coverage', xy=(3, 0.5), fontsize=9, color='gray', ha='center')
ax.annotate('No TEMPO coverage', xy=(20.5, 0.5), fontsize=9, color='gray', ha='center')

ax.set_xlabel('Local hour (Central)')
ax.set_ylabel('Normalized NO₂ (0–1 per series)')
ax.set_xticks(range(0, 24))
ax.legend(loc='lower left')
ax.set_title('Full diurnal cycle — surface NO₂ (24h), TEMPO (daylight), TROPOMI (overpass), 2024')
plt.tight_layout()
plt.show()

In [ ]:
# Same figure in absolute units: AQS in ppb (left axis), TEMPO column in real
# column units (right axis), TROPOMI as the timing line. Shows true magnitudes and
# the ~25% column swing vs ~2.7x surface swing that normalization hides.
fig, ax1 = plt.subplots(figsize=(11, 5.5))
ax1.plot(aqs_24h.index, aqs_24h.values, 's-', color='#333333', label='AQS surface NO\u2082 (all hours)')
ax1.set_xlabel('Local hour (Central)')
ax1.set_ylabel('AQS surface NO\u2082 (ppb)', color='#333333')
ax1.set_xticks(range(0, 24))
ax1.tick_params(axis='y', labelcolor='#333333')

ax2 = ax1.twinx()
ax2.plot(tempo_diurnal.index, tempo_diurnal.values, 'o-', color='#d62728',
         label='TEMPO column (daylight only)')
ax2.set_ylabel('TEMPO column (\u00d710\u00b9\u2074 molec/cm\u00b2)', color='#d62728')
ax2.tick_params(axis='y', labelcolor='#d62728')

ax1.axvline(trop_mean_hour, color='#1f77b4', linestyle=':', linewidth=2,
            label='TROPOMI overpass time (~12:30-13:30 local, one snapshot/day)')

ax1.axvspan(0, tempo_lo, color='gray', alpha=0.12)
ax1.axvspan(tempo_hi, 23, color='gray', alpha=0.12)
ax1.annotate('No TEMPO coverage', xy=(3, aqs_24h.max() * 0.7),
             fontsize=9, color='gray', ha='center')
ax1.annotate('No TEMPO coverage', xy=(20.5, aqs_24h.max() * 0.7),
             fontsize=9, color='gray', ha='center')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower left')

ax1.set_title('Full diurnal cycle (absolute units) \u2014 surface NO\u2082 vs TEMPO column, 2024')
plt.tight_layout()
plt.show()

In [ ]:
# ERA5 hourly boundary layer height (BLH) over the study area, full 2024
def fetch_era5_blh(year=2024):
    cache = DATA_DIR / f'era5_blh_{year}.csv'
    if cache.exists():
        return pd.read_csv(cache, parse_dates=['datetime_gmt'])

    month_frames = []
    for m in range(1, 13):
        month_cache = DATA_DIR / f'_era5_blh_{year}_{m:02d}.csv'
        if month_cache.exists():
            month_frames.append(pd.read_csv(month_cache))
            continue

        start = ee.Date.fromYMD(year, m, 1)
        ic = (ee.ImageCollection('ECMWF/ERA5/HOURLY')
              .filterDate(start, start.advance(1, 'month'))
              .select('boundary_layer_height'))

        def with_mean(img):
            stat = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=study_area,
                                     scale=25000, maxPixels=1e9)
            return img.set('blh', stat.get('boundary_layer_height'))

        rows = (ic.map(with_mean)
                .filter(ee.Filter.notNull(['blh']))
                .reduceColumns(reducer=ee.Reducer.toList(2),
                               selectors=['system:time_start', 'blh'])
                .get('list').getInfo())

        mdf = pd.DataFrame(rows, columns=['time_ms', 'blh_m'])
        mdf.to_csv(month_cache, index=False)
        month_frames.append(mdf)
        print(f'{year}-{m:02d}: {len(mdf)} hours')

    df = pd.concat(month_frames, ignore_index=True)
    df['datetime_gmt'] = pd.to_datetime(df['time_ms'], unit='ms', utc=True)
    df = df[['datetime_gmt', 'blh_m']].sort_values('datetime_gmt')
    df.to_csv(cache, index=False)
    for m in range(1, 13):
        (DATA_DIR / f'_era5_blh_{year}_{m:02d}.csv').unlink(missing_ok=True)
    return df


era5_blh = fetch_era5_blh(2024)
print(f'{len(era5_blh):,} hourly boundary layer heights')

In [ ]:
# Annual mean boundary layer depth by local hour, with the NO2 diurnal cycles for
# context: surface NO2 is the BLH curve inverted; the column ignores BLH entirely.
blh_hourly = (era5_blh
              .groupby(era5_blh['datetime_gmt'].dt.tz_convert(LOCAL_TZ).dt.hour)['blh_m']
              .mean())

tempo_diurnal = (paired.groupby('local_hour')['tempo_column'].mean())

fig, ax1 = plt.subplots(figsize=(10, 5.5))
ax1.fill_between(blh_hourly.index, blh_hourly.values, color='#85B7EB', alpha=0.45,
                 label='Boundary layer depth (ERA5)')
ax1.plot(blh_hourly.index, blh_hourly.values, color='#185FA5', linewidth=1.5)
ax1.set_xlabel('Local hour (Central)')
ax1.set_ylabel('Boundary layer depth (m)', color='#185FA5')
ax1.tick_params(axis='y', labelcolor='#185FA5')
ax1.set_xticks(range(0, 24))
ax1.set_ylim(0, blh_hourly.max() * 1.15)

ax2 = ax1.twinx()
ax2.plot(aqs_24h.index, aqs_24h.values, 's-', color='#333333', label='AQS surface NO₂')
ax2.set_ylabel('Surface NO₂ (ppb)', color='#333333')
ax2.tick_params(axis='y', labelcolor='#333333')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

ax1.set_title('Boundary layer depth vs surface NO₂ — 2024 annual mean by hour')
plt.tight_layout()
plt.show()

print('BLH range:', f"{blh_hourly.min():.0f} m (hour {blh_hourly.idxmin()})",
      '→', f"{blh_hourly.max():.0f} m (hour {blh_hourly.idxmax()})")
print('Correlation BLH vs surface NO₂ (hourly means):',
      f"{np.corrcoef(blh_hourly.reindex(aqs_24h.index), aqs_24h)[0, 1]:.3f}")